# Walmart Store Sales — Prophet v2: seasonal-residual correction

v1 external-covariate experiment was invalid for comparison because `bfill()` propagated later Markdown values backwards into early dates. This version removes all external regressors.

Instead of forecasting sales directly, Prophet forecasts the residual relative to the 52-week seasonal-naive value:

```text
residual(t) = sales(t) - sales(t - 52)
prediction(t) = sales(t - 52) + Prophet(residual(t))
```

This keeps the strongest known Walmart signal—the same Store-Dept week one year earlier—as the base forecast. Prophet only tries to learn a smaller correction from residual trend, yearly seasonality and holidays.

- validation is the final `39` weeks of `train.csv`;
- metric is Kaggle-style WMAE, holiday weight `5`;
- all `3,331` Store-Dept series are evaluated;
- no target leakage or future covariate imputation is used.


In [ ]:
%pip install -q "prophet>=1.1,<2" "wandb>=0.19,<1" "pandas>=2.2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4" "scikit-learn>=1.4,<2"


In [ ]:
from __future__ import annotations

import json
import logging
import os
import platform
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb
from prophet import Prophet

warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)

pd.set_option("display.max_columns", 120)
print({
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "wandb": wandb.__version__,
})


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"Not running in Colab or Drive unavailable: {exc}")


## Configuration

Prophet still fits one independent model per `(Store, Dept)` series. The only modelling change from the full baseline is the residual target. `residual_lag_weeks=52` defines the seasonal-naive base forecast.

In [ ]:
SEED = 42
np.random.seed(SEED)

CONFIG = {
    "seed": SEED,
    "validation_weeks": 39,
    "holiday_weight": 5.0,
    "top_n_series": None,
    "min_history_points": 52,
    "residual_lag_weeks": 52,
    "min_residual_history_points": 26,
    "growth": "linear",
    "yearly_seasonality": True,
    "weekly_seasonality": False,
    "daily_seasonality": False,
    "seasonality_mode": "additive",
    "changepoint_prior_scale": 0.05,
    "seasonality_prior_scale": 10.0,
    "holidays_prior_scale": 10.0,
    "interval_width": 0.80,
    "residual_clip_multiplier": 3.0,
    "prediction_clip_min": 0.0,
    "prediction_clip_max": 300000.0,
    "wandb_project": "Walmart-Recruiting---Store-Sales-Forecasting",
    "wandb_entity": "kende23-n-a",
    "wandb_group": "prophet-experiments",
    "run_name": "prophet_v2_seasonal_residual_all_series_validation",
    "artifact_name": "prophet-v2-seasonal-residual-all-series-validation",
}

DATA_DIR = Path("/content/drive/MyDrive/walmart_competition_data")
OUTPUT_DIR = Path("/content/artifacts/prophet_v2_seasonal_residual")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG

## Load data


In [ ]:
required_files = ["train.csv", "test.csv", "features.csv", "stores.csv"]
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError({"data_dir": str(DATA_DIR), "missing_files": missing_files})

train_raw = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["Date"])
test_raw = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["Date"])
features_raw = pd.read_csv(DATA_DIR / "features.csv", parse_dates=["Date"])
stores_raw = pd.read_csv(DATA_DIR / "stores.csv")

required_train = {"Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"}
required_test = {"Store", "Dept", "Date", "IsHoliday"}
missing = {
    "train": sorted(required_train.difference(train_raw.columns)),
    "test": sorted(required_test.difference(test_raw.columns)),
}
if any(missing.values()):
    raise ValueError(missing)

train_raw = train_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
test_raw = test_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

profile = pd.DataFrame({
    "table": ["train", "test", "features", "stores"],
    "rows": [len(train_raw), len(test_raw), len(features_raw), len(stores_raw)],
    "columns": [train_raw.shape[1], test_raw.shape[1], features_raw.shape[1], stores_raw.shape[1]],
    "min_date": [train_raw.Date.min(), test_raw.Date.min(), features_raw.Date.min(), pd.NaT],
    "max_date": [train_raw.Date.max(), test_raw.Date.max(), features_raw.Date.max(), pd.NaT],
})
display(profile)
display(train_raw.head())


## Split and all-series selection

In [ ]:
all_train_dates = pd.Index(sorted(train_raw["Date"].unique()), name="Date")
test_dates = pd.Index(sorted(test_raw["Date"].unique()), name="Date")
val_dates = all_train_dates[-CONFIG["validation_weeks"]:]
fit_dates = all_train_dates[:-CONFIG["validation_weeks"]]

if len(test_dates) != CONFIG["validation_weeks"]:
    raise ValueError(f"Expected test horizon {CONFIG['validation_weeks']}, got {len(test_dates)}")
if len(fit_dates) < CONFIG["min_history_points"]:
    raise ValueError("Not enough fit history for configured Prophet experiment.")
if len(fit_dates) - CONFIG["residual_lag_weeks"] < CONFIG["min_residual_history_points"]:
    raise ValueError("Not enough history after the 52-week residual lag.")

series_sales = (
    train_raw.groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
    .sum()
    .sort_values("Weekly_Sales", ascending=False)
)
if CONFIG["top_n_series"] is None:
    top_series = series_sales[["Store", "Dept"]].copy()
else:
    top_series = series_sales.head(int(CONFIG["top_n_series"]))[["Store", "Dept"]].copy()

selected_keys = set(map(tuple, top_series[["Store", "Dept"]].to_numpy()))
selected_train = train_raw.merge(top_series.assign(_keep=1), on=["Store", "Dept"], how="inner").drop(columns="_keep")

split_summary = {
    "validation_weeks": CONFIG["validation_weeks"],
    "fit_start": str(fit_dates.min().date()),
    "fit_end": str(fit_dates.max().date()),
    "validation_start": str(val_dates.min().date()),
    "validation_end": str(val_dates.max().date()),
    "test_start": str(test_dates.min().date()),
    "test_end": str(test_dates.max().date()),
    "all_train_rows": int(len(train_raw)),
    "selected_train_rows": int(len(selected_train)),
    "top_n_series": CONFIG["top_n_series"],
    "selected_series": int(len(selected_keys)),
    "residual_lag_weeks": CONFIG["residual_lag_weeks"],
}
display(pd.Series(split_summary, name="value").to_frame())

## Metric, holiday calendar and seasonal-naive reference

In [ ]:
def wmae(y_true, y_pred, is_holiday, holiday_weight: float = 5.0) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    weights = np.where(np.asarray(is_holiday, dtype=bool), holiday_weight, 1.0)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))


def make_holidays_frame(dates: pd.Series, is_holiday: pd.Series) -> pd.DataFrame:
    holidays = pd.DataFrame({
        "ds": pd.to_datetime(dates),
        "is_holiday": np.asarray(is_holiday, dtype=bool),
    })
    holidays = holidays.loc[holidays["is_holiday"], ["ds"]].drop_duplicates()
    holidays["holiday"] = "walmart_holiday"
    return holidays[["holiday", "ds"]]


holiday_calendar = pd.concat([
    train_raw[["Date", "IsHoliday"]].rename(columns={"Date": "ds"}),
    test_raw[["Date", "IsHoliday"]].rename(columns={"Date": "ds"}),
], ignore_index=True).drop_duplicates("ds")
holidays_df = make_holidays_frame(holiday_calendar["ds"], holiday_calendar["IsHoliday"])

holiday_by_date = (
    train_raw[["Date", "IsHoliday"]]
    .drop_duplicates("Date")
    .set_index("Date")
    .reindex(all_train_dates)["IsHoliday"]
    .fillna(False)
    .astype(bool)
)

sales_panel = (
    selected_train.pivot_table(index=["Store", "Dept"], columns="Date", values="Weekly_Sales", aggfunc="sum")
    .reindex(index=pd.MultiIndex.from_frame(top_series), columns=all_train_dates)
    .fillna(0.0)
    .sort_index()
)

actual_val = sales_panel.loc[:, val_dates].to_numpy(dtype=float)
seasonal_naive = sales_panel.loc[:, all_train_dates[-CONFIG["validation_weeks"] - 52:-52]].to_numpy(dtype=float)
val_holidays_matrix = np.tile(holiday_by_date.loc[val_dates].to_numpy(dtype=bool), (sales_panel.shape[0], 1))
seasonal_naive_wmae = wmae(actual_val.ravel(), seasonal_naive.ravel(), val_holidays_matrix.ravel(), CONFIG["holiday_weight"])

print({
    "selected_series": len(sales_panel),
    "seasonal_naive_wmae": seasonal_naive_wmae,
    "holiday_dates_known": len(holidays_df),
    "residual_lag_weeks": CONFIG["residual_lag_weeks"],
})
display(holidays_df.head())

## Fit Prophet on seasonal residuals

For each series, the first 52 fit weeks create the lag. The remaining fit weeks train Prophet on `sales(t) - sales(t-52)`. Validation reconstruction always starts from the real 52-week-lagged sales value, exactly as the seasonal-naive benchmark does.

In [ ]:
def fit_predict_prophet_for_series(store: int, dept: int, series: pd.Series) -> tuple[pd.DataFrame, dict]:
    lag = CONFIG["residual_lag_weeks"]
    fit_values = series.loc[fit_dates].astype(float).clip(lower=0.0)
    actual = series.loc[val_dates].astype(float).to_numpy()
    seasonal_fallback = series.loc[all_train_dates[-CONFIG["validation_weeks"] - lag:-lag]].astype(float).to_numpy()

    residual_values = fit_values.iloc[lag:].to_numpy(dtype=float) - fit_values.iloc[:-lag].to_numpy(dtype=float)
    history = pd.DataFrame({"ds": fit_dates[lag:], "y": residual_values})

    info = {
        "Store": int(store),
        "Dept": int(dept),
        "status": "fit",
        "sales_history_points": int(len(fit_values)),
        "residual_history_points": int(len(history)),
        "nonzero_residual_points": int((history["y"] != 0).sum()),
        "error": "",
    }

    if len(history) < CONFIG["min_residual_history_points"] or info["nonzero_residual_points"] < 2:
        residual_pred = np.zeros(len(val_dates), dtype=float)
        info["status"] = "fallback_insufficient_residual_history"
    else:
        try:
            model = Prophet(
                growth=CONFIG["growth"],
                yearly_seasonality=CONFIG["yearly_seasonality"],
                weekly_seasonality=CONFIG["weekly_seasonality"],
                daily_seasonality=CONFIG["daily_seasonality"],
                seasonality_mode=CONFIG["seasonality_mode"],
                changepoint_prior_scale=CONFIG["changepoint_prior_scale"],
                seasonality_prior_scale=CONFIG["seasonality_prior_scale"],
                holidays_prior_scale=CONFIG["holidays_prior_scale"],
                interval_width=CONFIG["interval_width"],
                holidays=holidays_df,
            )
            model.fit(history)
            forecast = model.predict(pd.DataFrame({"ds": val_dates}))
            residual_pred = forecast["yhat"].to_numpy(dtype=float)
            residual_limit = max(1000.0, float(np.nanmax(np.abs(residual_values))) * CONFIG["residual_clip_multiplier"])
            residual_pred = np.nan_to_num(residual_pred, nan=0.0, posinf=residual_limit, neginf=-residual_limit)
            residual_pred = np.clip(residual_pred, -residual_limit, residual_limit)
        except Exception as exc:
            residual_pred = np.zeros(len(val_dates), dtype=float)
            info["status"] = "fallback_fit_error"
            info["error"] = repr(exc)[:300]

    pred = seasonal_fallback + residual_pred
    pred = np.clip(pred, CONFIG["prediction_clip_min"], CONFIG["prediction_clip_max"])
    records = pd.DataFrame({
        "Store": int(store),
        "Dept": int(dept),
        "Date": pd.to_datetime(val_dates),
        "IsHoliday": holiday_by_date.loc[val_dates].to_numpy(dtype=bool),
        "Weekly_Sales": actual,
        "SeasonalNaive52": seasonal_fallback,
        "ResidualPrediction": residual_pred,
        "Prediction": pred,
        "AbsError": np.abs(actual - pred),
        "ModelStatus": info["status"],
    })
    return records, info

In [ ]:
start_time = time.time()
all_predictions = []
series_infos = []

for idx, ((store, dept), row) in enumerate(sales_panel.iterrows(), start=1):
    pred_df, info = fit_predict_prophet_for_series(int(store), int(dept), row)
    all_predictions.append(pred_df)
    series_infos.append(info)
    if idx % 25 == 0 or idx == len(sales_panel):
        elapsed = time.time() - start_time
        print({"finished_series": idx, "total_series": len(sales_panel), "elapsed_min": round(elapsed / 60, 2)})

val_pred_df = pd.concat(all_predictions, ignore_index=True)
series_info_df = pd.DataFrame(series_infos)

prophet_wmae = wmae(val_pred_df["Weekly_Sales"], val_pred_df["Prediction"], val_pred_df["IsHoliday"], CONFIG["holiday_weight"])
prophet_mae = float(np.mean(np.abs(val_pred_df["Weekly_Sales"] - val_pred_df["Prediction"])))
improvement_vs_seasonal = 100.0 * (seasonal_naive_wmae - prophet_wmae) / seasonal_naive_wmae

metrics = {
    "validation/wmae": float(prophet_wmae),
    "validation/mae": float(prophet_mae),
    "validation/seasonal_naive_wmae": float(seasonal_naive_wmae),
    "validation/improvement_vs_seasonal_naive_pct": float(improvement_vs_seasonal),
    "fit/series_total": int(len(series_info_df)),
    "fit/series_fit_ok": int((series_info_df["status"] == "fit").sum()),
    "fit/series_fallback": int((series_info_df["status"] != "fit").sum()),
    "fit/elapsed_minutes": float((time.time() - start_time) / 60),
}
print(metrics)
display(series_info_df["status"].value_counts().rename_axis("status").reset_index(name="count"))
display(val_pred_df.head())


## Diagnostics and W&B logging

In [ ]:
run = wandb.init(
    entity=CONFIG["wandb_entity"],
    project=CONFIG["wandb_project"],
    group=CONFIG["wandb_group"],
    name=CONFIG["run_name"],
    job_type="experiment_train",
    tags=["prophet", "classical-statistical", "experiment", "seasonal-residual", "all-series", "wmae"],
    config={**CONFIG, **split_summary},
    save_code=True,
)

val_pred_path = OUTPUT_DIR / "prophet_v2_validation_predictions.csv"
series_info_path = OUTPUT_DIR / "prophet_v2_series_info.csv"
metrics_path = OUTPUT_DIR / "prophet_v2_metrics.json"
config_path = OUTPUT_DIR / "prophet_v2_config.json"

val_pred_df.to_csv(val_pred_path, index=False)
series_info_df.to_csv(series_info_path, index=False)
metrics_path.write_text(json.dumps(metrics, indent=2))
config_path.write_text(json.dumps({**CONFIG, **split_summary}, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sample = val_pred_df.sample(min(8000, len(val_pred_df)), random_state=SEED)
axes[0].scatter(sample["Weekly_Sales"], sample["Prediction"], s=8, alpha=0.25)
max_axis = np.nanpercentile(sample[["Weekly_Sales", "Prediction"]].to_numpy(), 99)
axes[0].plot([0, max_axis], [0, max_axis], color="red", linewidth=1)
axes[0].set_title("Prophet v2 residual: actual vs prediction")
axes[0].set_xlabel("Actual Weekly_Sales")
axes[0].set_ylabel("Predicted Weekly_Sales")

weekly_errors = (
    val_pred_df.groupby("Date", as_index=False)
    .agg(Weekly_MAE=("AbsError", "mean"), IsHoliday=("IsHoliday", "max"))
)
axes[1].plot(weekly_errors["Date"], weekly_errors["Weekly_MAE"], marker="o")
for date in weekly_errors.loc[weekly_errors["IsHoliday"], "Date"]:
    axes[1].axvline(date, color="red", alpha=0.15)
axes[1].set_title("Validation MAE by week")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("MAE")
plt.tight_layout()
plot_path = OUTPUT_DIR / "prophet_v2_validation_diagnostics.png"
fig.savefig(plot_path, dpi=160)
plt.show()

artifact = wandb.Artifact(
    CONFIG["artifact_name"],
    type="model-evaluation",
    description="Prophet v2 seasonal-residual validation predictions and diagnostics. Per-series models are not serialized during experimentation.",
    metadata={**CONFIG, **metrics, **split_summary},
)
for artifact_path in [val_pred_path, series_info_path, metrics_path, config_path, plot_path]:
    artifact.add_file(str(artifact_path))
run.log_artifact(artifact, aliases=["v2", "validation", "latest"])

wandb.log({
    **metrics,
    "validation/prediction_table": wandb.Table(dataframe=val_pred_df.sample(min(20000, len(val_pred_df)), random_state=SEED)),
    "validation/series_info": wandb.Table(dataframe=series_info_df),
    "validation/weekly_errors": wandb.Table(dataframe=weekly_errors),
    "validation/diagnostic_plot": wandb.Image(str(plot_path)),
})
for key, value in metrics.items():
    run.summary[key] = value
run.summary["validation_predictions_path"] = str(val_pred_path)
run.summary["seasonal_naive_wmae"] = float(seasonal_naive_wmae)
run.summary["prophet_wmae"] = float(prophet_wmae)
run.summary["residual_lag_weeks"] = int(CONFIG["residual_lag_weeks"])

metrics

## End run

This experiment creates validation evidence only. No Kaggle submission is generated here.

In [ ]:
print("Experiment complete. W&B contains metrics, diagnostics, validation predictions and series status.")

In [ ]:
wandb.finish()
